# Notebook 03: Feature Engineering**Goal:** Create features that help the model learn better, based on EDA insights.Principle: Every feature must be justified by data evidence, not guesswork.

In [ ]:
import pandas as pdimport numpy as npimport syssys.path.insert(0, "../src")from housing.data.load_data import load_raw_data, load_configfrom housing.data.clean_data import clean_datafrom housing.features.build_features import engineer_features, get_feature_columns

In [ ]:
config = load_config("../config/config.yaml")df = load_raw_data("../data/raw/housing.csv")df_clean = clean_data(df)

## 1. Apply Feature Engineering

In [ ]:
df_features = engineer_features(df_clean)print(f"Original columns: {list(df_clean.columns)}")print(f"New columns added: {[c for c in df_features.columns if c not in df_clean.columns]}")print(f"\nShape: {df_features.shape}")

## 2. Inspect Engineered Features

In [ ]:
new_features = ['sqft_per_room', 'total_rooms', 'bed_bath_ratio', 'is_new',                 'size_category', 'age_category', 'price_per_sqft']print(df_features[new_features].head(10))

## 3. Validate CorrelationsCheck if engineered features actually add signal.

In [ ]:
numeric_cols = ['square_feet', 'bedrooms', 'bathrooms', 'age',                 'sqft_per_room', 'total_rooms', 'bed_bath_ratio', 'is_new', 'price']corr_with_price = df_features[numeric_cols].corr()['price'].sort_values(ascending=False)print(corr_with_price)

**Correlation Results:**| Feature | Correlation with Price ||---------|----------------------|| square_feet | 0.797 || sqft_per_room | 0.379 || total_rooms | 0.126 || bedrooms | 0.125 || bathrooms | 0.123 || is_new | 0.096 || age | -0.092 || bed_bath_ratio | -0.013 |`sqft_per_room` adds meaningful signal (0.38). The rest are weak but may help in combination with tree models.

## 4. Save Processed Data

In [ ]:
df_features.to_csv("../data/processed/housing_clean.csv", index=False)print("Saved processed data to ../data/processed/housing_clean.csv")

## Feature Engineering Decisions| Feature | Rationale | Keep? ||---------|-----------|-------|| `sqft_per_room` | Captures spaciousness beyond raw sqft | ✅ Yes (0.38 corr) || `total_rooms` | Reduces bedroom/bathroom multicollinearity | ✅ Yes || `bed_bath_ratio` | Luxury indicator | ⚠️ Marginal (-0.01 corr) || `is_new` | Slight premium for new homes in EDA | ✅ Yes || `size_category` | Non-linear size effects | ✅ For tree models || `age_category` | Non-linear age effects | ⚠️ Age is weak overall || `price_per_sqft` | For EDA only — **do not use in model** (data leakage) | ❌ Drop before modeling |